# Pelajaran 11 - Protokol Konteks Model (MCP)

The **Protokol Konteks Model (MCP)** ialah piawaian terbuka yang membolehkan ejen menemui dan menggunakan alat, sumber, dan sumber data secara dinamik semasa runtime. Daripada mengkodkan alat ke dalam ejen, MCP membolehkan ejen menyambung ke pelayan luaran yang mendedahkan kebolehan mengikut permintaan.

Dalam pelajaran ini, anda akan mempelajari:
- Apa itu MCP dan mengapa ia penting untuk sistem ejen
- Bagaimana seni bina klien-pelayan MCP berfungsi
- Bagaimana membina ejen yang menggunakan penemuan alat gaya MCP


## Persediaan

**Prasyarat:**
- Projek Azure AI Foundry dengan model yang telah disebarkan
- Jalankan `az login` untuk pengesahan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apakah Model Context Protocol (MCP)?

MCP mentakrifkan cara piawai bagi agen AI untuk menemui dan berinteraksi dengan alat luaran dan sumber data:

- **MCP Server**: Mendedahkan alat, sumber, dan prompt melalui protokol piawai
- **MCP Client**: Runtime agen yang menyambung kepada pelayan dan menemui keupayaan yang tersedia
- **Dynamic Discovery**: Agen tidak memerlukan alat yang ditakrifkan secara keras — mereka menemui apa yang tersedia semasa runtime

Ini sangat berguna untuk membina sistem agen yang boleh diperluas di mana keupayaan baru boleh ditambah tanpa mengubah suai kod agen.


## Bagaimana MCP Berfungsi

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ejen (klien MCP) menyambung ke pelayan MCP
2. Pelayan memberi respons dengan senarai alat yang tersedia dan skema mereka
3. Ejen kemudian boleh memanggil mana-mana alat yang ditemui semasa penalarannya
4. Keputusan mengalir kembali melalui protokol yang sama


## Mensimulasikan Penemuan Alat MCP

Oleh kerana pelayan MCP yang sebenar memerlukan proses pelayan yang sedang berjalan, kita akan menunjukkan corak ini menggunakan fungsi `@tool` yang mensimulasikan apa yang akan disediakan oleh perkhidmatan penginapan yang bersambung ke MCP.

Dalam persekitaran pengeluaran, alat-alat ini akan ditemui secara dinamik daripada pelayan MCP dan bukannya ditakrifkan secara tempatan.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Membina Ejen dengan Alat Gaya MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP dalam Pengeluaran

Dalam persekitaran pengeluaran, MCP membolehkan corak yang kuat:

- **Penemuan alat dinamik**: Ejen menyambung ke pelayan MCP dan menemui alat semasa program berjalan
- **Seni bina terpisah**: Pembekal alat boleh mengemas kini secara bebas daripada ejen
- **Perkongsian merentas organisasi**: Pasukan boleh mendedahkan keupayaan melalui pelayan MCP yang boleh digunakan oleh mana-mana ejen
- **Sokongan Microsoft Agent Framework**: MAF mempunyai sokongan klien MCP terbina dalam melalui integrasi `mcp`

Untuk menggunakan pelayan MCP sebenar dengan MAF, anda boleh menyambung melalui `hosted_mcp_tool()` atau integrasi klien MCP.

**Ketahui lebih lanjut:**
- [Spesifikasi MCP](https://modelcontextprotocol.io/)
- [Sokongan Microsoft Agent Framework untuk MCP](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Ringkasan

Dalam pelajaran ini, anda telah mempelajari:
- **MCP** adalah piawaian terbuka untuk penemuan alat dinamik antara ejen dan pembekal alat
- **Senibina klien-pelayan** membolehkan ejen menemui keupayaan pada masa larian
- MCP membolehkan **sistem ejen yang boleh dikembangkan dan terpisah** di mana alat boleh ditambah tanpa mengubah kod
- Microsoft Agent Framework menyediakan **sokongan MCP terbina dalam** untuk penggunaan pengeluaran


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan penterjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber rujukan utama yang sah. Bagi maklumat yang kritikal, penterjemahan profesional oleh penterjemah manusia adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
